In [1]:
import os

import numpy as np
import tensorflow as tf
import tqdm
from absl import app, flags, logging
from tqdm_multiprocess import TqdmMultiProcessPool

from droid.data_loading.trajectory_sampler import crawler
from droid.trajectory_utils.misc import load_trajectory

2026-02-12 09:45:48.109792: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-12 09:45:48.255626: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-12 09:45:48.260097: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/nvidia/lib:/usr/local/nv

In [5]:
def tensor_feature(value):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[tf.io.serialize_tensor(value).numpy()]))


def resize_and_encode(image, size):
    return tf.io.encode_jpeg(tf.cast(tf.round(tf.image.resize(image, size, method="bicubic")), tf.uint8))

def flatten(x):
    d = {}
    for k, v in x.items():
        if isinstance(v, dict):
            for k2, v2 in flatten(v).items():
                d[k + "/" + k2] = v2
        else:
            d[k] = v
    return d

FRAMESKIP = 1
IMAGE_SIZE = (180, 320)

KEEP_KEYS = [
    "action/cartesian_position",
    "action/cartesian_velocity",
    "action/gripper_position",
    "action/gripper_velocity",
    "action/target_cartesian_position",
    "action/target_gripper_position",
    "observation/image/24259877_left",
    "observation/image/24259877_right",
    "observation/image/13062452_left",
    "observation/image/13062452_right",
    "observation/image/20521388_left",
    "observation/image/20521388_right",
]

In [8]:
path = "/app/data/success/2026-02-11/Wed_Feb_11_09:18:01_2026"
output_path = "/app/data/tfrecords/train/0.tfrecord"
writer = tf.io.TFRecordWriter(output_path)


h5_filepath = os.path.join(path, "trajectory.h5")
# recording_folderpath = os.path.join(path, "recordings", "SVO")
recording_folderpath = os.path.join(path, "recordings", "MP4")

# this is really inefficient, should really do frame skipping inside `load_trajectory` but I didn't want to mess
# with it for now
traj = load_trajectory(h5_filepath, recording_folderpath=recording_folderpath)
traj = traj[::FRAMESKIP]



In [16]:
print(len(traj))
def print_keys(d, indent=0):
    """递归打印字典的所有 keys"""
    if isinstance(d, dict):
        for key, value in d.items():
            print(" " * indent + str(key))
            print_keys(value, indent + 2)
    elif isinstance(d, list):
        for i, item in enumerate(d):
            print(" " * indent + f"[{i}]")
            print_keys(item, indent + 2)

# 示例：打印 edge_all_open_tabs 的所有 keys
print_keys(traj[0])




493
action
  cartesian_position
  cartesian_velocity
  gripper_position
  gripper_velocity
  joint_position
  joint_velocity
  robot_state
    cartesian_position
    gripper_position
    joint_positions
    joint_torques_computed
    joint_velocities
    motor_torques_measured
    prev_command_successful
    prev_controller_latency_ms
    prev_joint_torques_computed
    prev_joint_torques_computed_safened
  target_cartesian_position
  target_gripper_position
observation
  camera_extrinsics
    17967083_left
    17967083_left_gripper_offset
    17967083_right
    17967083_right_gripper_offset
    19824535_left
    19824535_right
    23404442_left
    23404442_right
    26405488_left
    26405488_right
    29838012_left
    29838012_right
    31780776_left
    31780776_right
    36064187_left
    36064187_right
  camera_intrinsics
    17967083_left
    17967083_right
    31780776_left
    31780776_right
    36064187_left
    36064187_right
  camera_type
    17967083
    31780776
    3606

In [ ]:
# each element of `traj` is a possibly nested dict; flatten them and make sure they all have the same keys
traj_flat = [flatten(t) for t in traj]
assert all(t.keys() == traj_flat[0].keys() for t in traj_flat)

# convert to a single dict of lists, processing images and discarding unwanted keys along the way
out = {}
for key in traj_flat[0].keys():
    if key not in KEEP_KEYS:
        continue
    if "image" in key:
        out[key] = [resize_and_encode(t[key], IMAGE_SIZE) for t in traj_flat]
    else:
        out[key] = [t[key] for t in traj_flat]
example = tf.train.Example(features=tf.train.Features(feature={k: tensor_feature(v) for k, v in out.items()}))

writer.write(example.SerializeToString())


writer.close()